# Brain Tumor YOLO Classification — Google Colab Training

1. Make sure GPU is enabled: **Runtime -> Change runtime type -> T4 GPU**.
2. Upload `colab_training_bundle.zip` (either upload it to the Colab Files sidebar `/content/`, put it in your Google Drive `MyDrive/`, or upload when prompted in Step 2).
3. Run all cells (**Runtime -> Run all**).
4. At the end, `weights_output.zip` (`best.pt`, `confusion_matrix.png`, `evaluation_report.json`) will download automatically—extract it into your local `Mini_Project/weights/` folder and restart `app.py`.

In [ ]:
# Step 1: Verify GPU runtime and install Ultralytics
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, then re-run."
    )
print(f"GPU ready: {torch.cuda.get_device_name(0)}")
!pip install -q ultralytics matplotlib

In [ ]:
# Step 2: Locate and extract colab_training_bundle.zip
import zipfile
from pathlib import Path

project_dir = Path("/content/Mini_Project")
if (project_dir / "train.py").exists() and (project_dir / "Brain_tumor_dataset/content/Training").exists():
    print(f"Project already extracted at {project_dir}")
else:
    candidates = [
        Path("/content/colab_training_bundle.zip"),
        Path("/content/drive/MyDrive/colab_training_bundle.zip"),
    ]
    bundle_path = next((p for p in candidates if p.exists()), None)

    if bundle_path is None:
        mode = input("Type 'drive' to load colab_training_bundle.zip from Google Drive, or press Enter to upload from PC: ").strip().lower()
        if mode == "drive":
            from google.colab import drive
            drive.mount("/content/drive")
            matches = list(Path("/content/drive/MyDrive").rglob("colab_training_bundle.zip"))
            if not matches:
                raise FileNotFoundError("Could not find colab_training_bundle.zip in Google Drive (MyDrive).")
            bundle_path = matches[0]
        else:
            from google.colab import files
            uploaded = files.upload()
            if not uploaded:
                raise RuntimeError("No zip file uploaded.")
            bundle_path = Path.cwd() / next(iter(uploaded.keys()))

    project_dir.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {bundle_path} -> {project_dir} ...")
    with zipfile.ZipFile(bundle_path, "r") as zf:
        zf.extractall(project_dir)
    print("Extraction complete!")

In [ ]:
# Step 3: Train YOLOv8m-cls on Brain Tumor Dataset and Evaluate
%cd /content/Mini_Project
!python train.py --epochs 30 --batch 16 --imgsz 224 --patience 10

In [ ]:
# Step 4: Inspect Metrics & Download weights_output.zip
import json
import shutil
import zipfile
from pathlib import Path
from IPython.display import Image, display
from google.colab import files

weights_dir = Path("/content/Mini_Project/weights")
best_pt = weights_dir / "best.pt"
report_json = weights_dir / "evaluation_report.json"
cm_png = weights_dir / "confusion_matrix.png"

assert best_pt.exists(), f"Expected trained weights at {best_pt}"

if report_json.exists():
    print("=== Evaluation Report ===")
    print(json.dumps(json.loads(report_json.read_text()), indent=2))

if cm_png.exists():
    display(Image(filename=str(cm_png)))

out_zip = Path("/content/weights_output.zip")
with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for artifact in [best_pt, report_json, cm_png]:
        if artifact.exists():
            zf.write(artifact, arcname=artifact.name)

drive_root = Path("/content/drive/MyDrive")
if drive_root.exists():
    shutil.copy(out_zip, drive_root / "weights_output.zip")
    print(f"Also backed up to Google Drive: {drive_root / 'weights_output.zip'}")

print(f"Downloading {out_zip} ...")
files.download(str(out_zip))